# TrueCarry — Shot Browser + VLA Training
Colab pipeline. Loads frame sessions and Garmin truth **by Drive file ID** (from `MANIFEST.json`) — no folder wrangling. GPU (Colab Pro) used for training.

**Pipeline:** Swift replay harness exports per-frame `(u,v,diameter)` + ball-speed features → this notebook pairs them with Garmin VLA and trains. Feature CSVs go in Drive `TrueCarry-Training/features/`.

In [ ]:
!pip -q install gdown pillow scikit-learn pandas matplotlib
import gdown, json, zipfile, os, glob, re, io
import pandas as pd, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
os.makedirs('/content/data', exist_ok=True)
MANIFEST_ID = '145BzESan3TtLW71h0_alTmse4kEYmMqy'  # project folder; put MANIFEST.json id below
# paste the MANIFEST.json file id after uploading it:
MANIFEST_FILE_ID = '1DVLFKveT1tIGj69yMicJFXgquJRlfewS'
gdown.download(id=MANIFEST_FILE_ID, output='/content/MANIFEST.json', quiet=True)
M = json.load(open('/content/MANIFEST.json'))
pd.DataFrame(M['sessions'])[['date','local_time','conditions','ball','res','shots','garmin_shots']]

## 1 · Shot browser — pull any session and view its shots

In [ ]:
def load_session(date, local_time):
    s = next(x for x in M['sessions'] if x['date']==date and x['local_time']==local_time)
    out = f"/content/data/{date}_{local_time.replace(':','')}"
    if not os.path.exists(out):
        gdown.download(id=s['frames_zip_id'], output='/content/s.zip', quiet=False)
        with zipfile.ZipFile('/content/s.zip') as z: z.extractall(out)
    shots = sorted({p.split('/')[-2] for p in glob.glob(f'{out}/**/frame_*.jpg', recursive=True)})
    return out, shots

out, shots = load_session('2026-07-22','20:17')   # dark yellow
print(len(shots),'shots')

In [ ]:
def show_shot(out, shot, idxs=(15,18,20,22,26)):
    base = glob.glob(f'{out}/**/{shot}', recursive=True)[0]
    fig,ax = plt.subplots(1,len(idxs),figsize=(3*len(idxs),3))
    for a,i in zip(ax,idxs):
        f=f'{base}/frame_{i:03d}.jpg'
        if os.path.exists(f): a.imshow(Image.open(f)); a.set_title(f'f{i}')
        a.axis('off')
    plt.suptitle(shot); plt.tight_layout(); plt.show()

show_shot(out, shots[0])

## 2 · VLA training
Load the Swift-exported feature CSVs (`shot_id, t, u, v, diameter, ball_speed`) + the Garmin CSVs, pair by timestamp, and fit VLA. Start simple (gradient-boosted regressor on per-shot flight features), then iterate.

In [ ]:
# ---- load Garmin truth for a session ----
def load_garmin(garmin_csv_id):
    gdown.download(id=garmin_csv_id, output='/content/g.csv', quiet=True)
    g = pd.read_csv('/content/g.csv', skiprows=[1])   # row 1 is units
    g = g[pd.to_numeric(g['Ball Speed'], errors='coerce').notna()].copy()
    g['ball_speed']=g['Ball Speed'].astype(float); g['vla']=g['Launch Angle'].astype(float)
    g['t']=pd.to_datetime(g['Date']); return g[['t','ball_speed','vla']]

# ---- load Swift feature export (once generated: TrueCarry-Training/features/<session>.csv) ----
# feats = pd.read_csv(...)  # columns: shot_id,t,u,v,diameter,ball_speed
# Pair feats<->garmin by nearest timestamp, build per-shot flight features (diameter growth slope,
# vy, etc.), then:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
# X = per-shot features ; y = garmin vla
# model = HistGradientBoostingRegressor(max_depth=3)
# pred = cross_val_predict(model, X, y, cv=5)
# print('MAE', np.abs(pred-y).mean())
print('Wire up X,y once feature CSVs exist (see extract_features step in README).')